# **Makemore: Multi-Layer Perceptrons**
Scales the ideas from Micrograd and Makemore: Bigrams into a true multi-layer neural network with a hidden layer and non-linear activation functions. Model moves past and beyond static counting. Introduces data splitting, hyperparameter tuning, under and overfitting, etc. 

**Paper Followed:** A Neural Probabilistic Language Model - Bengio et al. 2003

## **Building Data-Set**

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# build vocab of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi= {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [8]:
block_size = 3 # context length: how many chars do we take to predict the next one
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


## **Embedding Look-Up Table**
Like the Weight matrix

In [4]:
# each 27 chars will have 2 dimensional embedding
# gives each character a (x, y) coordinate pair
C = torch.randn((27, 2))

In [5]:
C[5]

# Under the hood:
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([-0.0623, -1.4111])

In [6]:
C[[5, 6, 7]]

tensor([[-0.0623, -1.4111],
        [-1.5883, -0.4498],
        [ 0.8919,  1.6625]])

In [9]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

## **Hidden Layer**

In [ ]:
# 3 context characters with 2 embedding dimensions each
# and 100 hidden neurons:
W1 = torch.randn((6,100)) 
# bias term for each of the 100 neurons
b1 = torch.randn(100)

In [ ]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)
# torch.cat(torch.unbind(emb, 1), 1)

In [11]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [16]:
# PyTorch doesn't actually copy or move the underlying data in memory, it just changes the viewfinder
a.view(2, 9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [24]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 6) --> (-1, 6) so it's not hardcoded and program will determine dimensions on its own
h

tensor([[ 0.9694,  0.7562, -0.9018,  ..., -1.0000,  0.9322,  0.3697],
        [ 0.9890,  0.4242, -0.9264,  ..., -1.0000,  0.9615,  0.2316],
        [ 0.8566, -0.9829,  0.3116,  ..., -0.9999,  0.9981, -0.4172],
        ...,
        [-0.9958,  0.1060, -0.8321,  ...,  0.9949,  0.9398,  0.9994],
        [-0.9998,  0.9847, -0.9998,  ..., -0.2007, -1.0000,  0.5598],
        [-0.5973,  0.9900, -0.9999,  ..., -1.0000, -0.9976,  0.5545]])

## **Output Layer**

In [ ]:
# takes 100 features coming out of hidden layer
# projects them onto 27 dimensions, one for every letter plus dot token
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2
# represents how much the model "prefers" each of the 27 chars to come next
logits

In [28]:
counts = logits.exp()

In [29]:
prob = counts / counts.sum(1, keepdims=True)

In [31]:
prob[0].sum()

tensor(1.)

In [35]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [ ]:
# extracting what the model got correct
# for each 32 sample, extracts probability of predicting correct "Y" (next char in sequence after 3 context chars)
prob[torch.arange(32), Y]

tensor([4.0711e-08, 2.9532e-06, 5.1843e-07, 1.4867e-08, 7.8259e-09, 2.5117e-08,
        6.2161e-01, 1.1996e-15, 7.6924e-09, 1.0815e-05, 4.8874e-02, 8.3247e-11,
        3.0681e-07, 3.6825e-12, 2.0949e-02, 1.8878e-08, 2.2852e-16, 2.6970e-02,
        7.1988e-09, 1.5225e-05, 3.1542e-07, 5.2658e-03, 4.3010e-12, 4.1858e-15,
        2.2171e-10, 2.9596e-06, 1.4730e-10, 1.2495e-09, 1.5068e-08, 4.7424e-10,
        8.5790e-06, 1.1434e-08])

In [37]:
# negative log-likelihood loss
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.1582)

## **Respectable Version**

In [49]:
X.shape, Y.shape  # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [50]:
g = torch.Generator().manual_seed(2147483647)   # for reproducibility
C = torch.randn((27, 2), generator=g)           # embdedding table          | shape: (27 chars, 2 embedding features each)
W1 = torch.randn((6, 100), generator=g)         # hidden layer weights      | (6 input features, 100 hidden neurons)
b1 = torch.randn(100, generator=g)              # hidden layer bias         | 100 values, 1 for each hidden neuron
W2 = torch.randn((100, 27), generator=g)        # output layer weights      | (100 inputs coming in, 27 chars)
b2 = torch.randn(27, generator=g)               # output layer bias         | 27 values, one for each char
# optimizer - when training model, instead of individually updating each matrix, this list can tune everything at once
parameters = [C, W1, b1, W2, b2]                

In [51]:
sum(p.nelement() for p in parameters)  # number of parameters in total

3481

In [52]:
emb = C[X]  # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
logits = h @ W2 + b2  # (32, 27)

# counts = logits.exp()
# prob = counts / counts.sum(1, keepdims=True)
# loss = -prob[torch.arange(32), Y].log().mean()

# shortcut: doesn't create tensors that'll take up memory, backward pass will be more efficient
loss = F.cross_entropy(logits, Y) # arguements: (model's predictions, target labels)
loss

tensor(17.7697)

## **Training Loop**
Before training a model, check: can the model successfully overfit a single, tiny batch of data?

A model with 3,481 parameters should easily be able to memorize 32 examples. Loss should plummet all the way down to 0.0

In [53]:
for p in parameters:
    p.requires_grad = True

### **"Fitting these 32 examples"**
Using optimization (gradient descent) to tune model's parameters so that its error on that specific subset of data drops as close to zero as possible

In [ ]:
for _ in range(10):
    emb = C[X] 
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, Y) 
    print(loss.item())

# backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
# update
    for p in parameters:
        p.data += -0.1 * p.grad

# can't get to exactly zero | "..." --> predicting several chars

17.76971435546875
13.656400680541992
11.298768997192383
9.4524564743042
7.984262466430664
6.891321182250977
6.100014686584473
5.452036380767822
4.898151874542236
4.414664268493652


## **Training Full Dataset**